# Lab 1. Wine quality regression

Predict the expert grade `quality` (0–10 scale) of a wine from 11 laboratory measurements.

- **Data:** [UCI Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality), red wines only, 1599 rows.
- **Metric:** RMSE, required level **< 0.70**; MAE as a secondary metric.
- **Plan:** data and EDA → honest split → baseline → Linear Regression (plain and with a Yeo-Johnson transform), Random Forest, Gradient Boosting → comparison → feature importance → error analysis.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import KFold, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler

warnings.filterwarnings("ignore", message="The groups parameter is ignored")
pd.set_option("display.notebook_repr_html", False)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 72

SEED = 42
TARGET = "quality"
COLOR = "#b2182b"

## 1. Data

In [ ]:
wine = pd.read_csv("data/winequality-red.csv", sep=";")
wine.columns = wine.columns.str.replace(" ", "_")
FEATURES = [c for c in wine.columns if c != TARGET]

print(f"wines: {len(wine)}, features: {len(FEATURES)}")
print(f"missing values: {wine.isna().sum().sum()}, duplicate rows: {wine.duplicated().sum()}")
wine.describe().T[["mean", "std", "min", "50%", "max"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
sns.countplot(data=wine, x=TARGET, color=COLOR, ax=ax)
ax.set_title("Quality distribution")
plt.show()
wine[TARGET].value_counts(normalize=True).sort_index().round(3).to_frame("share").T

**Findings**

- All 11 features are numeric, no missing values — nothing to impute.
- Scales differ by orders of magnitude (std of `density` is 0.002, of `total_sulfur_dioxide` 33): linear coefficients will need standardization to be comparable; trees do not care.
- **240 rows (15%) are exact duplicates.** If copies of one wine get into both train and test, the model just remembers the answer, so the split must keep them together.
- **Long right tails:** `total_sulfur_dioxide` up to 289 (median 38), `chlorides` up to 0.611 (median 0.079), `residual_sugar` up to 15.5 (median 2.2), `sulphates` up to 2.0 (median 0.62). They look like real wines rather than errors, so they are kept. Trees are not affected by them; for linear regression they are influential points (see section 5).
- The target is **imbalanced**: grades 5–6 cover 83% of wines, grade 3 has 10 wines and grade 8 has 18. The models will see few extreme examples and will likely be weak on them.

## 2. Train/test split

- **80/20** (1279 / 320).
- **`StratifiedGroupKFold`**, first fold = test. *Stratified* by `quality`, so rare grades are in both parts in the same share. *Grouped* by identical rows, so duplicates never end up on both sides.
- `random_state=42` for reproducibility.
- The test is used **once**, at the end. All choices (hyperparameters, best model) are made by the same grouped stratified 5-fold CV on the training part.

In [ ]:
X, y = wine[FEATURES], wine[TARGET]
groups = wine.groupby(FEATURES, sort=False).ngroup()
CV = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(CV.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]
train = wine.iloc[train_idx]


def cv_rmse(model: object, cv: object = CV) -> tuple[float, float]:
    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
    )
    return -scores["train_score"].mean(), -scores["test_score"].mean()


shared = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"train: {len(train_idx)}, test: {len(test_idx)}, duplicates shared by both: {len(shared)}")

probe = RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=SEED)
print(
    f"CV RMSE with random KFold: {cv_rmse(probe, KFold(5, shuffle=True, random_state=SEED))[1]:.3f}"
)
print(f"CV RMSE with grouped CV:   {cv_rmse(probe)[1]:.3f}")

pd.DataFrame(
    {"train": y_train.value_counts(normalize=True), "test": y_test.value_counts(normalize=True)}
).sort_index().round(3).T

**Findings**

- Grade shares in train and test match, and no duplicate is shared between them. Grade 3 has 8 wines in train and 2 in test, grade 8 has 15 and 3.
- The same forest gets **0.580** with a random split of the training part and **0.641** with the grouped one. The only difference between the two runs is whether copies of a wine may be split between training and validation folds, so that 0.06 is remembered duplicates, not quality — the "unusually good result" the task warns about.
- **Why not evaluate many times on one test set:** if models and parameters are chosen by the test score, we gradually fit the noise of those particular 320 wines. The estimate becomes optimistic, and the test turns into a validation set. That is why every decision here is made by CV on train.

## 3. Exploratory analysis

Done on the **training part only**, so that no decision is based on the test answers.

In [ ]:
corr = train.corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, annot_kws={"size": 7}, ax=ax
)
ax.set_title("Pearson correlation (train)")
plt.show()

corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(3).to_frame(
    "corr with quality"
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, feature in zip(axes, ["alcohol", "volatile_acidity", "sulphates"], strict=True):
    sns.boxplot(data=train, x=TARGET, y=feature, color=COLOR, fliersize=1, ax=ax)
fig.tight_layout()
plt.show()
train[FEATURES].skew().sort_values(ascending=False).round(2).to_frame("skew")

**Findings**

- **`alcohol` is the strongest signal** (r = 0.49). Riper grapes have more sugar, which ferments into more alcohol and a fuller taste. But r² ≈ 0.24: alcohol explains less than a quarter of the variance, so the grade also depends on things that are not measured, and RMSE near zero is unrealistic.
- The alcohol effect is **non-linear**: the median stays around 10 for grades 3–5 and then grows (10.5 for grade 6, 11.5 for 7, 12.6 for 8).
- **`volatile_acidity`** (vinegar defect) is second (r = −0.39). Its median falls from 0.85 at grade 3 to 0.36 at grade 7 and stops falling at grade 8: high values mean "bad", but below some level lower is not better.
- **`sulphates`** (r = 0.25) grow with quality up to grade 7 and then flatten. The long right tail (a few wines above 1.5) pulls a straight line.
- `citric_acid` correlates with quality (0.23), but it is tied to volatile acidity (−0.54) and `fixed_acidity` (0.66), so it adds little new information. `density` is tied to `fixed_acidity` (0.66) and alcohol (−0.50). Such correlated features will make linear coefficients hard to interpret.
- `residual_sugar`, `free_sulfur_dioxide` and `pH` are almost unrelated to quality (|r| < 0.06).
- **Skewness:** `chlorides` (5.4), `residual_sugar` (4.6) and `sulphates` (2.4) are strongly right-skewed (a symmetric distribution has skew 0).
- Boxes of neighboring grades overlap heavily, so an error of about half a grade is normal even for a good model.

## 4. Baseline and metrics

**Baseline** is one constant for every wine. RMSE is minimized by the **mean** ($\frac{d}{dc}\sum (y_i - c)^2 = 0 \Rightarrow c = \bar y$), MAE by the median. The main metric is RMSE, so the baseline predicts the **mean of the training grades**.

$$\mathrm{MAE} = \frac{1}{n}\sum |y_i - \hat y_i| \qquad \mathrm{RMSE} = \sqrt{\frac{1}{n}\sum (y_i - \hat y_i)^2}$$

- **MAE** is the average size of the error in grade points ("off by half a grade on average").
- **RMSE** is the typical error, with large errors weighted more. It matches the MSE loss the models are trained on.
- **RMSE reacts more to large errors** because errors are squared: an error of 2 counts 4 times more than an error of 1, and one error of 3 equals nine errors of 1.
- **MAE ≤ RMSE always.** They are equal only when all errors are the same size. The gap grows when there are a few big mistakes among many small ones; here those are the rare grades.

## 5. Models

- **Linear Regression**: `StandardScaler` + `LinearRegression` in one pipeline, so the scaler is fitted on training data only. Scaling does not change the predictions, but it makes the coefficients comparable: each one is the effect of +1 standard deviation of its feature.
- **Linear Regression + Yeo-Johnson** (following [this article](https://www.dmitrymakarov.ru/data/transform-02/)): `PowerTransformer` picks a power for each feature that makes its distribution closer to normal, then standardizes it. For a long right tail this works like a logarithm: large values are compressed, so a few extreme wines pull the line less. Unlike Box-Cox, Yeo-Johnson accepts zeros (`citric_acid` has them). It is in the pipeline, so the powers are learned on training data only.
- **Random Forest**: the average of 500 deep trees, each trained on a bootstrap sample of rows; with `max_features` < 1 each split also chooses among a random subset of features. Averaging cancels out the mistakes of individual trees.
- **Gradient Boosting**: shallow trees built one after another, each correcting the errors of the previous ones with a small step (learning rate).

For each tree model, the default settings and one adjusted variant are compared by CV on train. The adjusted variants are standard textbook settings fixed in advance, not tuned on the test: `max_features="sqrt"` is the classic random-forest recommendation (more diverse trees), and for boosting a smaller learning rate with more trees, row subsampling and a minimum leaf size is the usual regularized recipe. Systematic tuning is the topic of lab 3.

In [ ]:
candidates = {
    "Baseline (mean)": DummyRegressor(strategy="mean"),
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Linear Regression + Yeo-Johnson": make_pipeline(PowerTransformer(), LinearRegression()),
    "RF default": RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=SEED),
    "RF max_features=sqrt": RandomForestRegressor(
        n_estimators=500, max_features="sqrt", n_jobs=-1, random_state=SEED
    ),
    "GB default": GradientBoostingRegressor(random_state=SEED),
    "GB 500 trees, lr=0.03, depth=5": GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        min_samples_leaf=5,
        random_state=SEED,
    ),
}
cv_table = pd.DataFrame(
    [cv_rmse(model) for model in candidates.values()],
    index=list(candidates),
    columns=["train RMSE", "CV RMSE"],
)
cv_table

- **RF with `max_features=sqrt`** (0.633) and **default boosting** (0.649) win their families on CV and are the ones evaluated on the test set.
- The adjusted boosting is worse than the default (0.655 vs 0.649): its train RMSE drops to 0.19 while CV grows. With only 1279 rows, 500 deeper trees start fitting noise.
- Linear regression has train ≈ CV (0.642 vs 0.652): it does not overfit. The Yeo-Johnson transform improves CV only slightly (0.652 → 0.649).
- Trees have train RMSE well below CV (RF 0.21 vs 0.63, GB 0.46 vs 0.65). For a forest this is normal (single trees memorize, averaging generalizes); only the CV score matters.

In [ ]:
def best_of(prefix: str) -> str:
    return cv_table.loc[cv_table.index.str.startswith(prefix), "CV RMSE"].idxmin()


models = {
    "Baseline": candidates["Baseline (mean)"],
    "Linear Regression": candidates["Linear Regression"],
    "Linear Regression + Yeo-Johnson": candidates["Linear Regression + Yeo-Johnson"],
    "Random Forest": candidates[best_of("RF")],
    "Gradient Boosting": candidates[best_of("GB")],
}
rows = []
for name, model in models.items():
    prediction = model.fit(X_train, y_train).predict(X_test)
    rows.append(
        {
            "model": name,
            "RMSE": root_mean_squared_error(y_test, prediction),
            "MAE": mean_absolute_error(y_test, prediction),
        }
    )
results = pd.DataFrame(rows).set_index("model")
results["beats baseline"] = results["RMSE"] < results.loc["Baseline", "RMSE"]
results["RMSE < 0.70"] = results["RMSE"] < 0.70
results

## 6. Comparison

All models use the same train, the same test and the same CV folds.

- **Every model beats the baseline** (0.799, which is roughly the spread of the grades, std 0.808). A median constant would give a slightly better MAE (0.653 instead of 0.678) but a worse RMSE.
- **Every model passes 0.70, including plain Linear Regression (0.657).** On red wine the main relations (alcohol, volatile acidity, sulphates) are close to monotonic, so a weighted sum of features describes them reasonably well.
- **Yeo-Johnson improves Linear Regression on the test from 0.657 to 0.645.** The coefficient of `sulphates` changes the most (section 7): once its tail is compressed, the relation becomes closer to linear. On CV the gain is only 0.003, so part of it may be chance.
- **Random Forest is the best (0.634)**, boosting is close (0.642), and the gap to the transformed linear model is only about 0.01. The grade is noisy (a subjective integer score, neighboring grades overlap) and the data is small, so a complex model cannot extract much more than a simple one.
- **Ceiling:** the best model removes only about 37% of the squared error of the constant baseline ($1 - (0.634/0.799)^2$). The rest depends on grape variety, aroma and the taste of the experts, none of which is in the data.

## 7. Feature importance

In [ ]:
linear, linear_yj, forest, boosting = (
    models[n]
    for n in [
        "Linear Regression",
        "Linear Regression + Yeo-Johnson",
        "Random Forest",
        "Gradient Boosting",
    ]
)
importance = pd.DataFrame(
    {
        "LR raw coef": LinearRegression().fit(X_train, y_train).coef_,
        "LR std coef": linear[-1].coef_,
        "LR YJ coef": linear_yj[-1].coef_,
        "RF": forest.feature_importances_,
        "GB": boosting.feature_importances_,
        "corr": corr[TARGET].drop(TARGET),
    },
    index=FEATURES,
).sort_values("RF", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
order = importance.index[::-1]
for ax, column, color in zip(
    axes, ["LR std coef", "RF", "GB"], ["#4393c3", "#5aae61", "#8073ac"], strict=True
):
    importance.loc[order, column].plot.barh(ax=ax, color=color)
    ax.set_title(column)
fig.tight_layout()
plt.show()
importance.round(3)

**Findings**

- **Raw linear coefficients are misleading.** `density` gets −11.5 only because its std is 0.002; after standardization it is just −0.02. The largest standardized coefficients are alcohol (+0.31), volatile acidity (−0.19), sulphates (+0.15) and total SO₂ (−0.11).
- **After Yeo-Johnson** the `sulphates` coefficient grows from 0.15 to 0.21: with the tail compressed, the model can use the relation. At the same time `density` and `fixed_acidity` get larger coefficients of opposite signs (−0.15 and +0.15). They are correlated (0.66) and compensate each other — **multicollinearity**, so these coefficients cannot be read one by one.
- `free_sulfur_dioxide` gets a positive linear coefficient despite ~0 correlation: `total_sulfur_dioxide` is also in the model, so the coefficient is the effect of free SO₂ at a fixed total.
- **RF and GB agree on the top:** `alcohol` first, then `volatile_acidity` and `sulphates`, then `total_sulfur_dioxide`, in line with the linear model and the correlations.
- **Difference:** `density` is 4th in RF (0.091) but low in GB (0.040). In a forest, alcohol is often missing from the random feature subset, so the tree splits on a correlated substitute. Boosting uses alcohol directly (0.368) and does not need it.
- `citric_acid` correlates with quality (0.23) but is barely used: its information is already in volatile acidity.

**Does feature importance mean that changing a feature changes the quality? No.**
Importance shows how much the model *uses* a feature to predict in this data. That is an association, not a cause. Alcohol is probably a marker of ripe grapes and a good vintage; adding spirit to a wine will not make it better. Correlated features also swap importance with each other (alcohol ↔ density). Proving a cause needs an experiment, for example a blind tasting of one wine at different alcohol levels.

## 8. Error analysis (Random Forest)

In [ ]:
errors = wine.iloc[test_idx].assign(prediction=forest.predict(X_test))
errors["error"] = errors[TARGET] - errors["prediction"]
levels = sorted(errors[TARGET].unique())

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
sns.histplot(errors["error"], bins=40, color="#4393c3", ax=axes[0])
axes[0].set_title("Error = true - predicted")
sns.boxplot(data=errors, x=TARGET, y="prediction", color="#92c5de", fliersize=2, ax=axes[1])
axes[1].plot(range(len(levels)), levels, "r--", label="ideal")
axes[1].legend()
axes[1].set_title("Prediction by true quality")
fig.tight_layout()
plt.show()

within = errors["error"].abs()
print(f"mean error: {errors['error'].mean():.3f}, median |error|: {within.median():.2f}")
print(f"|error| < 0.5: {(within < 0.5).mean():.0%}, |error| < 1: {(within < 1).mean():.0%}")
print(f"predictions range: {errors['prediction'].min():.2f} - {errors['prediction'].max():.2f}")
errors.groupby(TARGET).agg(
    count=("error", "size"),
    mean_prediction=("prediction", "mean"),
    MAE=("error", lambda e: e.abs().mean()),
)

In [ ]:
errors.reindex(errors["error"].abs().nlargest(5).index)[
    [TARGET, "prediction", "alcohol", "volatile_acidity", "sulphates", "total_sulfur_dioxide"]
]

**Findings**

- Errors are centered at zero (mean −0.027) and roughly symmetric. 63% of wines are predicted within ±0.5 of the true grade, 90% within ±1.
- **Predictions are squeezed towards the middle:** true grades range over 3–8, predictions only over 4.57–7.17. The model overestimates bad wines and underestimates good ones. For grades 5–6 the MAE is 0.39; for grade 3 it is about 1.9, for grades 4 and 8 about 1.25. The test has only 2 wines of grade 3 and 3 of grade 8.
- **Why rare grades are worse:**
  1. *Few examples:* train has 8 wines of grade 3 and 15 of grade 8, so even in the "bad" or "good" regions the leaves are dominated by grades 5–6.
  2. *The loss pulls to the mean:* minimizing MSE predicts the average grade of similar wines, and with overlapping grades that average is close to the middle.
  3. *Averaging:* the forest averages 500 trees, which smooths out extreme predictions even more.
- **Worst cases:**
  - #7: true 7, predicted 4.8. By every key feature it looks like a poor wine: alcohol 10 (median of grade 7 is 11.5), volatile acidity 0.65 (good wines have about 0.36), sulphates 0.47 (good wines about 0.75). The reason is something not measured, or the grading itself.
  - #903 / #904: the same wine twice, true 7, predicted 5.4. Average alcohol (10.8), high volatile acidity (0.59), low sulphates (0.59): the model sees an average wine, the experts rated it higher.
  - #1505: true 3, predicted 5.1. High volatile acidity (0.76) lowered the prediction, but alcohol and sulphates are typical of grade 5, so it stopped far from 3.
  - #1124: true 4, predicted 5.9. Here high alcohol (11.5) pulled the prediction up.

## Conclusions

| Model | CV RMSE | Test RMSE | Test MAE |
| --- | --- | --- | --- |
| Baseline | 0.809 | 0.799 | 0.678 |
| Linear Regression | 0.652 | 0.657 | 0.506 |
| Linear Regression + Yeo-Johnson | 0.649 | 0.645 | 0.496 |
| **Random Forest** | **0.633** | **0.634** | **0.491** |
| Gradient Boosting | 0.649 | 0.642 | 0.493 |

- **Final model: Random Forest, test RMSE 0.634 < 0.70.** It was chosen by CV on train; the test was used once.
- Every model passes the threshold, including Linear Regression; the Yeo-Johnson transform improves it slightly by taming the long-tailed features.
- Duplicates (15% of the data) would inflate the result by about 0.06 with a random split. The grouped split keeps the estimate honest.
- Key features: `alcohol`, `volatile_acidity` and `sulphates`. Feature importance shows association, not cause.
- The model is accurate on typical wines (grades 5–6) and pulls rare grades towards the middle.